# 05 — Simulation-First FPGA, JTAG, Flash, and LED Bring-up

**Modules:** `02.01`, `07.01`, `07.02`, `07.03`  
**Prerequisite:** notebooks 01–04; bring-up reuses the UART trace and exact system image rather than inventing a new payload.  
**Consumes:** `02-uart-trace.json`, `04-system-image.json`  
**Emits:** `fromthetransistor/05-bringup-report.json`

The journey is inspired by the historical [`realhw` branch](https://github.com/geohot/fromthetransistor/tree/realhw), which explored a Spartan-6 LX9 LED build and USB/SPI flash access. No code is copied: the branch has no LICENSE. Here, a fresh deterministic model turns the journey into a test ladder—protocol state, identity, bounded flash readback, UART trace, then LED state—before any claim about physical hardware.

In [ ]:
from pathlib import Path
import hashlib
import json
import os

ARTIFACT_ROOT = Path(os.environ['COURSE_NOTEBOOK_ARTIFACTS'])
uart = json.loads((ARTIFACT_ROOT / 'fromthetransistor/02-uart-trace.json').read_text(encoding='utf-8'))
system = json.loads((ARTIFACT_ROOT / 'fromthetransistor/04-system-image.json').read_text(encoding='utf-8'))
machine_image = bytes.fromhex(system['machine_image_hex'])
assert uart['schema_version'] == system['schema_version'] == 1
assert all(len(waveform) == 10 * uart['samples_per_bit'] for waveform in uart['waveforms'])
assert machine_image == bytes.fromhex(system['boot_block_hex'])[:len(machine_image)]


## Worked bring-up ladder

The TAP model checks a specific JTAG state path; the SPI flash checks command, chip-select transaction boundary, identity, address width, and read range; the LED counter checks reset and clocked state. Passing these simulations narrows software and protocol errors. It does **not** validate board revision, pin constraints, voltage rails, clock quality, timing closure, signal integrity, cable firmware, or the actual flash part.

Current-toolchain caveat: Spartan-6 normally requires legacy Xilinx ISE rather than current Vivado flows. Host compatibility and vendor-tool availability can change. Treat historical scripts and part numbers as archaeology until reverified against the exact board and current official documentation.

In [ ]:
TAP = {
    ('RESET', 0): 'IDLE', ('RESET', 1): 'RESET',
    ('IDLE', 0): 'IDLE', ('IDLE', 1): 'SELECT_DR',
    ('SELECT_DR', 0): 'CAPTURE_DR', ('SELECT_DR', 1): 'SELECT_IR',
    ('CAPTURE_DR', 0): 'SHIFT_DR', ('CAPTURE_DR', 1): 'EXIT1_DR',
    ('SHIFT_DR', 0): 'SHIFT_DR', ('SHIFT_DR', 1): 'EXIT1_DR',
    ('EXIT1_DR', 0): 'PAUSE_DR', ('EXIT1_DR', 1): 'UPDATE_DR',
    ('UPDATE_DR', 0): 'IDLE', ('UPDATE_DR', 1): 'SELECT_DR',
}

def tap_walk(start, tms_bits):
    state = start
    trace = [state]
    for bit in tms_bits:
        try:
            state = TAP[(state, bit)]
        except KeyError as exc:
            raise ValueError(f'unmodeled TAP transition from {state} with TMS={bit}') from exc
        trace.append(state)
    return trace

class SPIFlash:
    JEDEC_ID = bytes.fromhex('20ba18')
    def __init__(self, size=256):
        self.memory = bytearray([0xFF] * size)
    def preload(self, address, data):
        if address < 0 or address + len(data) > len(self.memory):
            raise ValueError('preload outside flash')
        self.memory[address:address + len(data)] = data
    def transact(self, tx, read_length):
        if not tx:
            raise ValueError('empty chip-select transaction')
        if tx[0] == 0x9F:
            if len(tx) != 1 or read_length != 3:
                raise ValueError('invalid JEDEC-ID transaction')
            return self.JEDEC_ID
        if tx[0] == 0x03:
            if len(tx) != 4:
                raise ValueError('read command needs a 24-bit address')
            address = int.from_bytes(tx[1:], 'big')
            if address + read_length > len(self.memory):
                raise ValueError('read outside flash')
            return bytes(self.memory[address:address + read_length])
        raise ValueError('unsupported flash opcode')

def led_counter(cycles, divider=4, reset_cycles=2):
    count = 0
    trace = []
    for cycle in range(cycles):
        reset = cycle < reset_cycles
        count = 0 if reset else count + 1
        trace.append({'cycle': cycle, 'reset': reset, 'count': count, 'led': (count // divider) & 1})
    return trace

tap_trace = tap_walk('RESET', [0, 1, 0, 0, 1, 1, 0])
assert tap_trace == ['RESET', 'IDLE', 'SELECT_DR', 'CAPTURE_DR', 'SHIFT_DR', 'EXIT1_DR', 'UPDATE_DR', 'IDLE']
flash = SPIFlash()
flash.preload(0x20, machine_image)
jedec = flash.transact(b'\x9f', 3)
readback = flash.transact(b'\x03\x00\x00\x20', len(machine_image))
assert jedec == SPIFlash.JEDEC_ID
assert readback == machine_image
led_trace = led_counter(18)
assert all(sample['count'] == 0 for sample in led_trace[:2])
assert {sample['led'] for sample in led_trace[2:]} == {0, 1}
image_hash = hashlib.sha256(machine_image).hexdigest()
assert hashlib.sha256(readback).hexdigest() == image_hash
checks = {
    'tap_path': True,
    'flash_identity': jedec.hex() == SPIFlash.JEDEC_ID.hex(),
    'flash_readback': readback == machine_image,
    'uart_trace_shape': all(len(row) == 40 for row in uart['waveforms']),
    'led_reset_and_toggle': all(sample['count'] == 0 for sample in led_trace[:2]) and len({sample['led'] for sample in led_trace[2:]}) == 2,
}
assert all(checks.values())
artifact = {
    'schema_version': 1,
    'inspiration': 'https://github.com/geohot/fromthetransistor/tree/realhw',
    'original_model_notice': 'No unlicensed branch code copied; simulation model authored for this suite.',
    'tap_trace': tap_trace,
    'jedec_id_hex': jedec.hex(),
    'image_sha256': image_hash,
    'checks': checks,
    'led_trace': led_trace,
    'physical_unknowns': ['board revision', 'pin constraints', 'voltage rails', 'clock integrity', 'timing closure', 'signal integrity', 'cable firmware', 'flash part'],
    'toolchain_caveat': 'Spartan-6 is an ISE-era family; reverify current supported hosts and tools.',
}
target = ARTIFACT_ROOT / 'fromthetransistor/05-bringup-report.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(artifact, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('simulation-first bring-up checks:', checks)


## Exercise and physical-boundary checklist

1. Predict the TAP state after each TMS bit before running the trace. Add one unmodeled IR path and require a diagnostic rather than guessing.
2. Change one flash address byte and show that readback hash fails.
3. Require an out-of-range flash read to fail before data transfer.
4. Write a physical bring-up decision tree that starts with rails, reference clock, reset, and identity before attempting a full image. Never infer electrical safety from this notebook.

In [ ]:
wrong_read = flash.transact(b'\x03\x00\x00\x21', len(machine_image))
assert hashlib.sha256(wrong_read).hexdigest() != image_hash
try:
    flash.transact(b'\x03\x00\x00\xf8', 16)
    raise AssertionError('out-of-range flash read accepted')
except ValueError:
    pass
try:
    tap_walk('IDLE', [1, 1, 0])
    raise AssertionError('unmodeled TAP path accepted')
except ValueError:
    pass
assert artifact['physical_unknowns']
print('bring-up boundary self-checks passed; simulation is not physical validation')
